In [ ]:
# updated training code to get all MFCCs of single wav to produce CDS
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import random
import gc
import shutil
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast, GradScaler
from tqdm.notebook import tqdm
from scipy.stats import spearmanr
from google.colab import runtime

# --- 1. Paths & Config ---
MFCC_DIR = '/content/mfcc_all_flat/'
SAVE_DIR = '/content/drive/MyDrive/BUU_PHD_THESIS/VIVIT_TRAINED_MODELS_MULTI_MFCC_ENGLISH_ONLY/'
TRAIN_CSV = '/content/drive/MyDrive/BUU_PHD_THESIS/train_split_english_only.csv'
VAL_CSV = '/content/drive/MyDrive/BUU_PHD_THESIS/val_split_english_only.csv'

RESUME_PATH = os.path.join(SAVE_DIR, 'vivit_cds_resume.pth')
EMERGENCY_PATH = os.path.join(SAVE_DIR, 'vivit_cds_emergency.pth')
BEST_PATH = os.path.join(SAVE_DIR, 'best_cds_vivit.pth')

# REDUCED BATCH SIZE: Long sequences use significantly more VRAM
BATCH_SIZE = 8
LEARNING_RATE = 1e-5 # Lower LR for stable CDS learning
NUM_EPOCHS = 500
EARLY_STOP_PATIENCE = 25
SYNC_INTERVAL = 200

# --- 2. Model: Sequence-Aware ViViT ---
class CDS_ViViT(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        self.proj = nn.Conv2d(1, embed_dim, kernel_size=(10, 31), stride=(10, 31))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)
        self.mos_head = nn.Linear(embed_dim, 1)

    def forward(self, x, mask=None):
        # x: [B, 40, Total_Frames]
        x = x.unsqueeze(1)
        x = self.proj(x).flatten(2).transpose(1, 2) # [B, Total_Tokens, 768]

        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) # [B, Total_Tokens + 1, 768]

        if mask is not None:
            # Shift mask for CLS token
            cls_mask = torch.zeros((mask.shape[0], 1), device=mask.device, dtype=torch.bool)
            full_mask = torch.cat([cls_mask, mask], dim=1)
            x = self.transformer(x, src_key_padding_mask=full_mask)
        else:
            x = self.transformer(x)

        return self.mos_head(x[:, 0]) # Prediction from the file-level CLS (CDS)

# --- 3. Dataset & Collate Function (The Stacking Logic) ---
class WholeFileDataset(Dataset):
    def __init__(self, df, base_dir):
        self.df = df
        self.base_dir = base_dir

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        audio_id = str(self.df.iloc[idx, 0]).replace('.wav', '')
        mos_label = float(self.df.iloc[idx, 1])
        folder_path = os.path.join(self.base_dir, audio_id)

        try:
            meta_df = pd.read_csv(os.path.join(folder_path, "segment_info.csv")).sort_values('segment_path')

            mfccs, masks = [], []
            for _, row in meta_df.iterrows():
                mfcc = np.load(os.path.join(folder_path, row['segment_path']))
                mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-6)
                mfccs.append(torch.from_numpy(mfcc).float())

                # Per-segment mask logic
                valid_slices = int(np.ceil(int(row['valid_frames']) / 31))
                m = torch.zeros((4, 10), dtype=torch.bool)
                m[:, valid_slices:] = True
                masks.append(m.flatten())

            # Concatenate all segments into one long MFCC sequence
            full_mfcc = torch.cat(mfccs, dim=1) # [40, Total_Frames]
            full_mask = torch.cat(masks) # [Total_Tokens]
            return full_mfcc, torch.tensor([mos_label]).float(), full_mask
        except:
            return self.__getitem__(random.randint(0, len(self.df)-1))

def collate_cds(batch):
    """Pads variable length audio files to the same length in a batch"""
    mfccs, labels, masks = zip(*batch)
    # Pad MFCCs: (Channels, Time) -> (B, Channels, Max_Time)
    padded_mfccs = pad_sequence([m.T for m in mfccs], batch_first=True, padding_value=0.0).transpose(1, 2)
    # Pad Masks: (Tokens) -> (B, Max_Tokens)
    padded_masks = pad_sequence(masks, batch_first=True, padding_value=True) # True = Ignore padding
    return padded_mfccs, torch.stack(labels), padded_masks

# --- 4. Training Engine ---
def validate(model, val_loader, device, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for inputs, labels, masks in val_loader:
            inputs, labels, masks = inputs.to(device), labels.to(device), masks.to(device)
            preds = model(inputs, mask=masks)
            total_loss += criterion(preds, labels).item()
            all_preds.extend(preds.cpu().numpy().flatten()); all_labels.extend(labels.cpu().numpy().flatten())
    srcc, _ = spearmanr(all_preds, all_labels)
    return total_loss / len(val_loader), srcc

def run_pretraining():
    os.makedirs(SAVE_DIR, exist_ok=True)
    device = torch.device("cuda")

    # Using the Custom Collate Function for variable-length files
    train_loader = DataLoader(WholeFileDataset(pd.read_csv(TRAIN_CSV), MFCC_DIR),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate_cds)
    val_loader = DataLoader(WholeFileDataset(pd.read_csv(VAL_CSV), MFCC_DIR),
                            batch_size=BATCH_SIZE, num_workers=2, collate_fn=collate_cds)

    model = CDS_ViViT().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss(); scaler = GradScaler('cuda')

    # Robust Load Logic
    start_epoch, best_mse, history = 0, float('inf'), []
    ckpt_path = RESUME_PATH if os.path.exists(RESUME_PATH) else EMERGENCY_PATH if os.path.exists(EMERGENCY_PATH) else None
    if ckpt_path:
        try:
            ckpt = torch.load(ckpt_path, map_location='cpu')
            model.load_state_dict(ckpt['model_state_dict']); optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch, history = ckpt['epoch'], ckpt.get('history', [])
            best_mse = min([h['val_loss'] for h in history]) if history else float('inf')

        except: print("⚠️ Checkpoint corrupted.")

    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train(); total_train_loss = 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch}")
        for i, (inputs, labels, masks) in enumerate(loop):
            inputs, labels, masks = inputs.to(device), labels.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda'):
                preds = model(inputs, mask=masks); loss = criterion(preds, labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_train_loss += loss.item()
            if (i + 1) % SYNC_INTERVAL == 0:
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'history': history}, EMERGENCY_PATH)

        val_mse, srcc = validate(model, val_loader, device, criterion)
        train_mse = total_train_loss / len(train_loader)
        history.append({'epoch': epoch, 'train_loss': train_mse, 'val_loss': val_mse, 'srcc': srcc})

        if val_mse < best_mse:
            best_mse = val_mse; torch.save(model.state_dict(), BEST_PATH)
            print(f"🌟 Best MSE: {best_mse:.4f}")

        torch.save({'epoch': epoch+1, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(), 'history': history}, RESUME_PATH)
        gc.collect(); torch.cuda.empty_cache()

if __name__ == "__main__":
    run_pretraining()